*  DSC 530
*  Weeks 5 & 6
  
*  Doug Nolan

# pasting all exercises here to use later

1. Exercise 1.
With the earthquakes.csv file, select all the earthquakes in Japan with a magnitude of 4.9 or greater using the mb magnitude type.

2. Exercise 2.
Create bins for each full number of earthquake magnitude (for instance, the first bin is (0,1], the second is (1,2], and so on) with the ml magnitude type and count how many are in each bin.

3. Exercise 4.
Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum magnitude that was observed for each combination. Put the magnitude type along the columns.

4. Exercise 6.
Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.

5. Exercise 7.
Calculate the z-scores for each numeric column of Amazon’s data (ticker is AMZN) in Q4 2018 using apply().

# Exercise 10 - covid file
1. Prepare the data
2. Read in the data in the covid19_cases.csv file.
3. Create a date column by parsing the dateRep column into a datetime.
4. Set the date column as the index.
5. Use the replace() method to update all occurrences of United_States_of_America and United_Kingdom to USA and UK, respectively.
6. Sort the index.
7. For the five countries with the most cases (cumulative), find the day with the largest number of cases.
8. Find the 7-day average change in COVID-19 cases for the last week in the data for the five countries with the most cases.
9. Find the first date that each country other than China had cases.
10. Rank the countries by cumulative cases using percentiles.

# Exercise 1

With the earthquakes.csv file, select all the earthquakes in Japan with a magnitude of 4.9 or greater using the mb magnitude type.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# read in csv file
# get only desired earthquakes. larger than 4.9 mag and only mb magTypes and only in Japan
earthquakes = pd.read_csv("Bellevue/dsc530/Hands-On-Data-Analysis-with-Pandas-2nd-edition-master/ch_04/exercises/earthquakes.csv")
print(earthquakes.shape)

# place is tricky and is a full address. parse and get the last word to use as country 
earthquakes['parsed_place'] = earthquakes['place'].str.split().str[-1]

big_earthquakes = earthquakes[(earthquakes['mag'] >= 4.9) & (earthquakes['magType'] == 'mb')
& (earthquakes['parsed_place'] == 'Japan')]
print(big_earthquakes.shape)
print(big_earthquakes)

# earthquakes.query("parsed_place == 'Japan' and magType == 'mb' and mag >= 4.9"
                 # )[['mag', 'magType', 'place']]

# big_earthquakes.head()

(9332, 6)
(4, 6)
      mag magType           time                         place  tsunami  \
1563  4.9      mb  1538977532250  293km ESE of Iwo Jima, Japan        0   
2576  5.4      mb  1538697528010    37km E of Tomakomai, Japan        0   
3072  4.9      mb  1538579732490     15km ENE of Hasaki, Japan        0   
3632  4.9      mb  1538450871260    53km ESE of Hitachi, Japan        0   

     parsed_place  
1563        Japan  
2576        Japan  
3072        Japan  
3632        Japan  


# Exercise 2

In [3]:
# create bins for only ml earthquake types. start at 0 and increase step size by 1 
import pandas as pd
import numpy as np

# typing earthquakes takes ages
df = earthquakes

# 1. Filter for 'ml' magnitude type
ml_earthquakes = df[df['magType'] == 'ml'].copy()

# 2. Define bins for each full number (0 to max magnitude)
max_mag = int(np.ceil(ml_earthquakes['mag'].max()))
bins = list(range(0, max_mag + 1))

# 3. Create bins (0, 1], (1, 2], etc.
ml_earthquakes['mag_bin'] = pd.cut(ml_earthquakes['mag'], bins=bins, right=True)

# 4. Count the number of earthquakes in each bin
bin_counts = ml_earthquakes['mag_bin'].value_counts().sort_index()

print(bin_counts)


mag_bin
(0, 1]    2207
(1, 2]    3105
(2, 3]     862
(3, 4]     122
(4, 5]       2
(5, 6]       1
Name: count, dtype: int64


# Exercise 4

In [4]:
crosstab = pd.crosstab(
    index=df['tsunami'],
    columns=df['magType'],
    values=df['mag'],
    aggfunc='max')
# crosstab function via pandas :) 
print(crosstab)

magType   mb  mb_lg    md   mh   ml  ms_20    mw  mwb  mwr  mww
tsunami                                                        
0        5.6    3.5  4.11  1.1  4.2    NaN  3.83  5.8  4.8  6.0
1        6.1    NaN   NaN  NaN  5.1    5.7  4.41  NaN  NaN  7.5


# Exercise 6

In [6]:
# yes, i spelled fang wrong on purpose, my brain thinks the extra a is wrong instead
fang = pd.read_csv("Bellevue/dsc530/Hands-On-Data-Analysis-with-Pandas-2nd-edition-master/ch_04/exercises/faang.csv")

fang.head()
# fang.dtypes

# pivot table comparing stocks - showw averages of OHLC (open, high, low, close) and volume 
fang.pivot_table(values=['open', 'high', 'low', 'close', 'volume'],
    index='ticker',
    aggfunc='mean')

,close,high,low,open,volume
ticker,,,,,
AAPL,47.263357,47.748526,46.795877,47.277859,1.360803e+08
AMZN,1641.726176,1662.839839,1619.840519,1644.072709,5.648994e+06
FB,171.510956,173.613347,169.303148,171.472948,2.765860e+07
GOOG,1113.225134,1125.777606,1101.001658,1113.554101,1.741965e+06
NFLX,319.290319,325.219322,313.187330,319.620558,1.146962e+07


In [7]:
fang.head()


,ticker,date,high,low,open,close,volume
0,FB,2018-01-02,181.580002,177.550003,177.679993,181.419998,18151900.0
1,FB,2018-01-03,184.779999,181.330002,181.880005,184.669998,16886600.0
2,FB,2018-01-04,186.210007,184.100006,184.899994,184.330002,13880900.0
3,FB,2018-01-05,186.899994,184.929993,185.589996,186.850006,13574500.0
4,FB,2018-01-08,188.899994,186.330002,187.199997,188.279999,17994700.0


# Exercise 7

In [9]:
# create filter for Q4 2018
# i hate python dates datetimes objects >.<

df = fang

# Filter for Amazon (AMZN)
amzn = df[df["ticker"] == "AMZN"]

# Filter for Q4 2020 (Oct 1 – Dec 31)
amzn_q4_2018 = amzn[
    (amzn["date"] >= "2018-10-01") & 
    (amzn["date"] <= "2018-12-31")
]
amzn_q4_2018.head()

# Select only numeric columns
numeric_cols = amzn_q4_2018.select_dtypes(include=["number"])

# # Calculate z-scores using apply()
z_scores = numeric_cols.apply(lambda x: (x - x.mean()) / x.std())

# # Optional: attach back to original filtered data
amzn_q4_2018_z = amzn_q4_2018.copy()
amzn_q4_2018_z[numeric_cols.columns] = z_scores

# # View result
print(amzn_q4_2018_z.head())

    ticker        date      high       low      open     close    volume
690   AMZN  2018-10-01  2.368006  2.502113  2.337813  2.385848 -1.630411
691   AMZN  2018-10-02  2.227302  2.247433  2.190795  2.155037 -0.861879
692   AMZN  2018-10-03  2.058955  2.139987  2.068570  2.025489 -0.920345
693   AMZN  2018-10-04  1.819474  1.781561  1.850048  1.722816 -0.126582
694   AMZN  2018-10-05  1.628173  1.554416  1.642819  1.584748 -0.298771


# Exercise 10


#### do these things
1. Prepare the data
2. Read in the data in the covid19_cases.csv file.
3. Create a date column by parsing the dateRep column into a datetime.
4. Set the date column as the index.
5. Use the replace() method to update all occurrences of United_States_of_America and United_Kingdom to USA and UK, respectively.
6. Sort the index.
7. For the five countries with the most cases (cumulative), find the day with the largest number of cases.
8. Find the 7-day average change in COVID-19 cases for the last week in the data for the five countries with the most cases.
9. Find the first date that each country other than China had cases.
10. Rank the countries by cumulative cases using percentiles.

In [12]:
#read pesky excel file, not CSV
# covid = pd.read_excel('Bellevue/dsc530/COVID-19-geographic-disbtribution-worldwide-2020-12-14.xlsx')
covid = pd.read_csv("Bellevue/dsc530/Hands-On-Data-Analysis-with-Pandas-2nd-edition-master/ch_04/exercises/covid19_cases.csv")
covid.head()

,dateRep,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
0,19/09/2020,19,9,2020,47,1,Afghanistan,AF,AFG,38041757.0,Asia,1.616645
1,18/09/2020,18,9,2020,0,0,Afghanistan,AF,AFG,38041757.0,Asia,1.535155
2,17/09/2020,17,9,2020,17,0,Afghanistan,AF,AFG,38041757.0,Asia,1.653446
3,16/09/2020,16,9,2020,40,10,Afghanistan,AF,AFG,38041757.0,Asia,1.708649
4,15/09/2020,15,9,2020,99,6,Afghanistan,AF,AFG,38041757.0,Asia,1.627159


In [13]:
# ensure date datatype is compatible for sorting
covid['date'] = pd.to_datetime(covid['dateRep'], dayfirst=True)

covid = covid.set_index('date').sort_index()

covid = covid.replace({
    'United_States_of_America': 'USA',
    'United_Kingdom': 'UK'})

covid.head()


,dateRep,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
date,,,,,,,,,,,,
2019-12-31,31/12/2019,31,12,2019,0,0,Belgium,BE,BEL,11455519.0,Europe,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Mexico,MX,MEX,127575529.0,America,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Ecuador,EC,ECU,17373657.0,America,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Russia,RU,RUS,145872260.0,Europe,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Netherlands,NL,NLD,17282163.0,Europe,NaN


In [14]:
# create sum cases column
covid["sum_cases"] = covid.groupby("countriesAndTerritories")["cases"].cumsum()

In [15]:
top_five_countries = covid\
    .groupby('countriesAndTerritories').cases.sum()\
    .nlargest(5).index

covid[covid.countriesAndTerritories.isin(top_five_countries)]\
    .groupby('countriesAndTerritories').cases.idxmax()

countriesAndTerritories
Brazil   2020-07-30
India    2020-09-17
Peru     2020-08-17
Russia   2020-07-18
USA      2020-07-25
Name: cases, dtype: datetime64[ns]

In [18]:
# 7 day average cahnge for the last week . top 5 countries again
covid_top5 = covid\
    .groupby(['countriesAndTerritories', pd.Grouper(freq='1D')]).cases.sum()\
    .unstack(0).diff().rolling(7).mean().last('1W')[top_five_countries]


print(covid_top5)



countriesAndTerritories          USA        India       Brazil      Russia  \
date                                                                         
2020-09-14                473.714286   181.285714    35.285714   36.285714   
2020-09-15               1513.000000  1142.857143   697.428571   46.285714   
2020-09-16               3478.714286    59.571429  3196.285714   61.428571   
2020-09-17              -1047.000000   308.428571   143.428571  810.000000   
2020-09-18                865.714286   -18.142857  -607.714286 -688.428571   
2020-09-19                306.857143  -604.714286  -560.142857   57.285714   

countriesAndTerritories        Peru  
date                                 
2020-09-14                73.142857  
2020-09-15               377.571429  
2020-09-16               -65.000000  
2020-09-17               -29.428571  
2020-09-18              -227.571429  
2020-09-19               -41.285714  


/var/folders/l0/w4cn9b_s39l3p2c89ztml7zw0000gn/T/ipykernel_44672/402320311.py:4: FutureWarning: last is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  .unstack(0).diff().rolling(7).mean().last('1W')[top_five_countries]


In [20]:
# # finding last 7 days average of the top 5 countries (via cum sum) -- incorrect - did not do average for last 7 days, but average for last day only 
# top5 = (
#     covid.groupby("countriesAndTerritories")["cases"]
#     .sum()
#     .nlargest(5)
#     .index
# )
# # print(top5)
# covid_top5 = covid[covid["countriesAndTerritories"].isin(top5)].copy()
# # sort for rolling average accuracy
# covid_top5 = covid_top5.sort_index()

# covid_top5["cases_7d_avg"] = (
#     covid_top5.groupby("countriesAndTerritories")["cases"]
#     .transform(lambda x: x.rolling(7).mean())
# )

# last_week = (
#     covid_top5.groupby("countriesAndTerritories")
#     .tail(7)
# )

# result = (
#     covid_top5.groupby("countriesAndTerritories")
#     .tail(1)[["countriesAndTerritories", "cases_7d_avg"]]
# )

# print(result)

In [24]:
#getting min date for all countries besides china
first_cases = (
    covid[
        (covid["countriesAndTerritories"] != "China") & 
        (covid["cases"] > 0)
    ]
    .reset_index()   # bring 'date' back as a column
    .groupby("countriesAndTerritories")["date"]
    .min()
    .reset_index(name="first_case_date")
)
print(first_cases)

    countriesAndTerritories first_case_date
0               Afghanistan      2020-02-25
1                   Albania      2020-03-09
2                   Algeria      2020-02-26
3                   Andorra      2020-03-03
4                    Angola      2020-03-22
..                      ...             ...
204                 Vietnam      2020-01-24
205          Western_Sahara      2020-04-26
206                   Yemen      2020-04-10
207                  Zambia      2020-03-19
208                Zimbabwe      2020-03-21

[209 rows x 2 columns]


In [25]:
#ranking countries based on sum cases column
country_totals = (
    covid.groupby("countriesAndTerritories")["sum_cases"]
    .max()
    .reset_index()
)

country_totals["percentile_rank"] = (
    country_totals["sum_cases"]
    .rank(pct=True)
)

# Optional: sort descending
country_totals = country_totals.sort_values("percentile_rank", ascending=False)

print(country_totals)

         countriesAndTerritories  sum_cases  percentile_rank
196                          USA    6724667         1.000000
92                         India    5308014         0.995238
27                        Brazil    4495183         0.990476
159                       Russia    1091186         0.985714
152                         Peru     756412         0.980952
..                           ...        ...              ...
79                     Greenland         14         0.023810
131                   Montserrat         13         0.016667
66   Falkland_Islands_(Malvinas)         13         0.016667
88                      Holy_See         12         0.009524
5                       Anguilla          3         0.004762

[210 rows x 3 columns]
